<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats_v2/corrections/seance1_correction.ipynb)

# Séance 3.1 — Décrire une distribution

**Correction** · durée : 4h (2h de cours, 2h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qu'une moyenne décrit — et ce qu'elle ne décrit pas
- choisir entre moyenne et médiane selon la forme de la distribution
- lire un `describe()` ligne par ligne
- mesurer la dispersion avec l'écart-type et l'écart interquartile
- repérer une concentration : quelle part du total tient dans le haut du classement
- décrire une variable qualitative — et savoir ce qu'on n'a pas le droit d'y calculer
- utiliser `groupby` pour calculer et comparer des statistiques entre plusieurs groupes
- utiliser `.agg()` pour calculer plusieurs indicateurs par groupe

## Exercice — L'estimateur de l'agence

## La question

Vous travaillez toujours pour la même agence parisienne. Le mois dernier, vous
avez répondu au client qui disposait de 400 000 € : le 19e, 51 m². La direction
a lu votre note, et elle en veut plus.

> *« Faites-nous un outil d'estimation. On tape un arrondissement et une
> surface, il rend une fourchette de prix. »*

Une **fourchette**, pas un prix. C'est tout le sujet de cet exercice. La
médiane que vous avez calculée la dernière fois ne suffit pas : elle dit où
est le milieu, elle ne dit rien de **l'écart entre les ventes**. Deux
arrondissements peuvent avoir la même médiane et se comporter très
différemment — dans l'un, presque toutes les ventes sont proches du milieu ;
dans l'autre, elles vont du simple au triple.

À la fin de cet exercice, vous aurez :

- une **fiche par arrondissement** : effectif, moyenne, médiane, écart-type, quartiles ;
- la réponse à « où les prix sont-ils les plus dispersés ? », et une explication ;
- un estimateur qui marche, que vous testerez sur une vraie annonce.

## Comment ça marche

**Chaque exercice est une cellule vide que vous écrivez entièrement.** Juste
avant, un encadré *Rappel* nomme les outils dont vous avez besoin. Vous n'avez
rien à deviner : vous avez à appliquer le cours à ce fichier.

Vous rencontrerez aussi trois autres sortes de cellules :

- des **cellules à exécuter telles quelles** : le code est déjà écrit, il vous montre quelque chose ;
- des **cellules de vérification**, aux moments où une erreur fausserait la suite. Elles affichent `OK` ou `A REVOIR` avec un indice ;
- des **cellules de prédiction** : on vous demande d'écrire ce que vous attendez, en commentaire, *avant* d'exécuter la cellule suivante.

Si une vérification affiche `NameError`, c'est que la cellule d'exercice
au-dessus n'a pas été exécutée, ou qu'elle contient une faute. Corrigez-la,
relancez-la, puis relancez la vérification.

## Partie 0 — Mise en route

Exécutez les trois cellules suivantes. Ce sont les mêmes qu'au bloc 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/bloc2_donnees_v2/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
ventes = pd.read_csv(BASE + "immo_paris_2024.csv")
ventes["date"] = pd.to_datetime(ventes["date"])   ## deja au format international : rien a redresser

print(ventes.shape)
ventes.head(3)

### D'où vient ce fichier

C'est **le vôtre**. C'est exactement ce que produisait le nettoyage de
« Où acheter à Paris ? » : les 25 209 ventes d'appartements de 2024 qui ont
survécu aux sept défauts, une ligne par vente, avec le prix au m² déjà
calculé. On vous le rend propre pour que vous n'ayez pas à refaire le travail.

| Colonne | Contenu |
|---|---|
| `vente_id` | l'identifiant de la vente |
| `date` | la date de la vente |
| `prix` | le prix payé, en euros |
| `rue` | le nom de la rue |
| `arrondissement` | de 1 à 20 |
| `surface` | la surface, en m² |
| `pieces` | le nombre de pièces principales |
| `longitude`, `latitude` | la position sur la carte |
| `prix_m2` | le prix divisé par la surface |

Une différence avec la dernière fois : la colonne `date` est au format
international (`2024-01-02`), donc une seule écriture et aucun piège. La
conversion tient en une ligne, elle est déjà faite ci-dessus.

---

## Partie 1 — Une ligne au lieu d'une boucle

### Exercice 1 — La photo d'ensemble

Affichez le `describe()` de `prix_m2`, arrondi à une décimale. Mettez la
moyenne dans `moyenne_paris` et la médiane dans `mediane_paris`.

Puis, en commentaire : **la moyenne dépasse la médiane de plus de 500 €.
Pourquoi ?** Le `max` de la colonne vous donne la réponse.

> **Rappel.** `describe()` pour les huit nombres, `mean()` et `median()` pour
> les deux qui nous intéressent.

In [ ]:
print(ventes["prix_m2"].describe().round(1))

moyenne_paris = ventes["prix_m2"].mean()
mediane_paris = ventes["prix_m2"].median()

print("moyenne", round(moyenne_paris), "| mediane", round(mediane_paris))

# La distribution est etiree vers la droite : le max est a 30 000 euros le m2, soit
# trois fois la mediane. Ces ventes tres cheres tirent la moyenne vers le haut,
# alors que la mediane ne bouge pas.

In [ ]:
verifier("la moyenne", round(moyenne_paris) == 10201, "mean() sur la colonne prix_m2")
verifier("la mediane", round(mediane_paris) == 9676, "median() sur la colonne prix_m2")

**10 201 € contre 9 676 €.** Vous avez rencontré cet écart au cours 3.1 sur
le panier des commandes : c'est la signature d'une distribution étirée vers la
droite. La moitié centrale des ventes va de 8 167 à 11 522 € le m² — c'est
déjà une fourchette, mais elle vaut pour Paris entier. La direction veut la
même chose, arrondissement par arrondissement.

### Exercice 2 — Vingt médianes, en une ligne

À l'exercice précédent, vous aviez calculé les vingt médianes avec une boucle
`for`, deux listes et un `pd.Series`. Six lignes. Le cours 3.1 les remplace par
une seule.

Calculez la médiane de `prix_m2` par arrondissement dans `medianes`, affichez
le résultat trié du moins cher au plus cher.

> **Rappel.** `groupby` sur la colonne qui définit les paquets, la colonne à
> résumer entre crochets, l'indicateur à la fin.

In [ ]:
medianes = ventes.groupby("arrondissement")["prix_m2"].median()

print(medianes.sort_values().round(0))

In [ ]:
verifier("vingt arrondissements", len(medianes) == 20, "groupby('arrondissement') : un paquet par arrondissement")
verifier("la mediane du 19e", round(medianes.loc[19]) == 7836, "la colonne a resumer est prix_m2, l'indicateur est median()")

**7 836 € le m² dans le 19e.** À l'euro près le nombre que votre boucle avait
produit. Ce que vous aviez écrit en six lignes, `groupby` le fait en une — et
vous savez ce qu'il y a dedans, puisque vous l'avez écrit à la main : découper,
calculer, recoller.

### Exercice 3 — Le creux d'août, en une ligne aussi

Même chose sur le temps. Comptez les ventes par mois dans `ventes_mois`, puis
mettez le numéro du mois le plus calme dans `mois_calme`.

> **Rappel.** `.dt.month` donne le mois d'une colonne de dates. Pour compter
> les lignes de chaque paquet plutôt que résumer une colonne : `size()`.
> `idxmin()` rend l'étiquette de la plus petite valeur.

In [ ]:
ventes_mois = ventes.groupby(ventes["date"].dt.month).size()
mois_calme = ventes_mois.idxmin()

print(ventes_mois)
print("mois le plus calme :", mois_calme)

In [ ]:
verifier("douze mois", len(ventes_mois) == 12, "groupby sur ventes['date'].dt.month")
verifier("le mois le plus calme", mois_calme == 8, "idxmin() rend l'etiquette, pas la valeur")

Août, comme la dernière fois. Vous venez de refaire en deux lignes ce qui vous
avait pris une partie de l'après-midi.

---

## Partie 2 — La fiche de chaque arrondissement

Une médiane par arrondissement, c'est un chiffre par arrondissement. Pour une
fourchette, il en faut plusieurs : combien de ventes, où est le milieu, et
jusqu'où les prix s'écartent de ce milieu.

`agg` calcule plusieurs indicateurs d'un coup, en donnant leur liste.

### Exercice 4 — Quatre indicateurs d'un coup

Construisez `fiche` : une ligne par arrondissement, et quatre colonnes dans
cet ordre — l'effectif, la moyenne, la médiane, l'écart-type de `prix_m2`.
Affichez le résultat arrondi.

L'effectif en premier n'est pas une coquetterie : c'est la règle du cours 3.1.
On ne commente pas un groupe sans savoir combien de ventes il contient.

> **Rappel.** `agg` prend la **liste** des indicateurs voulus, écrits entre
> guillemets : `"count"`, `"mean"`, `"median"`, `"std"`.

In [ ]:
fiche = ventes.groupby("arrondissement")["prix_m2"].agg(["count", "mean", "median", "std"])

fiche.round(0)

In [ ]:
verifier("une ligne par arrondissement", len(fiche) == 20, "groupby('arrondissement')")
verifier("quatre colonnes", list(fiche.columns) == ["count", "mean", "median", "std"],
         "agg(['count', 'mean', 'median', 'std']) : la liste, dans cet ordre")
verifier("la mediane du 19e", round(fiche.loc[19, "median"]) == 7836, "c'est la meme qu'a l'exercice 2")

Regardez la colonne `count` avant tout le reste. Le 1er arrondissement tient
en **248 ventes** sur l'année, le 15e en **2 678**. Dix fois plus. Tout ce
qu'on dira du 1er sera dix fois plus fragile que ce qu'on dira du 15e, et
c'est un point sur lequel l'exercice suivant reviendra longuement.

### Une colonne calculée à partir d'un groupby

Il manque les quartiles. `agg` ne sait pas les calculer directement, parce
qu'un quantile a besoin d'un argument : lequel ? On les ajoute donc comme des
colonnes, une par une :

```python
fiche["q25"] = ventes.groupby("arrondissement")["prix_m2"].quantile(0.25)
```

À droite, un `groupby` qui rend une Series : vingt valeurs, étiquetées par
numéro d'arrondissement. À gauche, une nouvelle colonne de `fiche`, dont les
lignes portent **les mêmes étiquettes**. pandas les met en face les unes des
autres tout seul. C'est le `df["c"] = ...` que vous connaissez, avec un
alignement par étiquette en plus.

### Exercice 5 — Les quartiles, et la largeur de la fourchette

Ajoutez à `fiche` trois colonnes : `q25`, `q75`, et `iqr` qui est leur
différence. Affichez la fiche entière.

> **Rappel.** `quantile(0.25)` et `quantile(0.75)` — une **proportion** entre
> 0 et 1, jamais un pourcentage. L'écart interquartile est une soustraction
> entre deux colonnes déjà présentes.

In [ ]:
fiche["q25"] = ventes.groupby("arrondissement")["prix_m2"].quantile(0.25)
fiche["q75"] = ventes.groupby("arrondissement")["prix_m2"].quantile(0.75)
fiche["iqr"] = fiche["q75"] - fiche["q25"]

fiche.round(0)

In [ ]:
verifier("le premier quartile du 6e", round(fiche.loc[6, "q25"]) == 12353, "quantile(0.25), pas quantile(25)")
verifier("l'ecart interquartile du 6e", round(fiche.loc[6, "iqr"]) == 5366, "iqr = q75 - q25")

**Lisez la ligne du 6e.** La moitié centrale de ses ventes va de 12 353 à
17 719 € le m². Pour un 40 m², cela fait une fourchette de
**494 118 à 708 744 €** — et cette fourchette ne couvre que la moitié
des ventes, un quart sont en dessous et un quart au-dessus.

« Le prix du 6e » n'existe pas. C'est ce que l'estimateur va devoir dire.

### Exercice 6 — Où les prix sont-ils les plus dispersés ?

L'écart-type ne se compare pas d'un arrondissement à l'autre : il est en euros,
donc un arrondissement cher a mécaniquement un écart-type plus grand. Pour
comparer, on le rapporte à la moyenne. C'est le **coefficient de variation**,
et il n'a pas d'unité.

Ajoutez la colonne `cv` à `fiche`, puis affichez les arrondissements du plus
dispersé au moins dispersé.

> **Rappel.** Le coefficient de variation est l'écart-type divisé par la
> moyenne. `sort_values` trie, `ascending=False` du plus grand au plus petit.

In [ ]:
fiche["cv"] = fiche["std"] / fiche["mean"]

print(fiche["cv"].sort_values(ascending=False).round(2))

In [ ]:
verifier("le coefficient de variation du 8e", round(fiche.loc[8, "cv"], 2) == 0.37, "std / mean, sur la fiche")
verifier("le moins disperse", fiche["cv"].idxmin() == 12, "idxmin() rend l'etiquette de la plus petite valeur")

En tête, le **8e** (0,37) et le **1er** (0,33). En queue, le **12e**
(0,22) et le **20e** (0,24) : le coefficient du 8e vaut
**1,7 fois** celui du 12e, et cet écart s'explique.

Le 8e mélange des hôtels particuliers de l'avenue Montaigne et des chambres de
bonne de trente mètres carrés sous les toits. Le 12e est fait de rues entières
d'immeubles comparables. **Les arrondissements chers sont les plus
hétérogènes** : c'est là que la médiane seule trompe le plus, et là que
l'estimateur devra annoncer la fourchette la plus large.

---

## Partie 3 — Ce que la moyenne ne dit pas

### Exercice 7 — Où est l'argent ?

Le cours 3.1 a posé la question sur le chiffre d'affaires du détaillant : quelle
part du total tient dans le haut du classement ? Ici, la question devient :
**quelle part de l'argent dépensé en 2024 va aux ventes de plus d'un million
d'euros ?**

Écrivez votre prédiction dans la cellule ci-dessous avant de calculer.

In [ ]:
# Ma prediction (part des euros qui va aux ventes de plus d'un million) :

Calculez maintenant `part_ventes`, le pourcentage des ventes qui dépassent un
million d'euros, et `part_euros`, le pourcentage du total dépensé qu'elles
représentent. Arrondissez à une décimale.

> **Rappel.** `query` pour isoler ces ventes, `len` pour les compter, `sum()`
> sur la colonne `prix` pour l'argent. Une part en pourcentage, c'est 100 fois
> la partie divisée par le tout.

In [ ]:
gros = ventes.query("prix > 1000000")

part_ventes = round(100 * len(gros) / len(ventes), 1)
part_euros = round(100 * gros["prix"].sum() / ventes["prix"].sum(), 1)

print(part_ventes, "% des ventes font", part_euros, "% des euros")

In [ ]:
verifier("part des ventes", part_ventes == 12.2, "len() sur les ventes filtrees, divise par len(ventes)")
verifier("part des euros", part_euros == 37.9, "sum() sur la colonne prix, pas len()")

**12,2 % des ventes, 37,9 % des euros.** Une vente sur huit porte plus du
tiers de l'argent qui change de main à Paris dans l'année.

Ce chiffre ne se lit dans aucune moyenne, et ce n'est pas que la moyenne le
cacherait : elle répond à une autre question. Pour une agence, c'est pourtant
**la** question, puisque la commission se calcule sur le prix.

### Exercice 8 — Le studio est-il cher au mètre carré ?

L'intuition du métier dit oui : plus c'est petit, plus le mètre carré se paie
cher. Écrivez votre prédiction, puis vérifiez.

In [ ]:
# Ma prediction (au m2, le studio est-il plus cher que le 3 pieces ?) :

Construisez `par_pieces` : pour chaque nombre de pièces, l'effectif et la
médiane de `prix_m2`. Affichez le tableau entier.

> **Rappel.** Le même `groupby(...).agg([...])` qu'à l'exercice 4, sur une
> autre colonne de regroupement.

In [ ]:
par_pieces = ventes.groupby("pieces")["prix_m2"].agg(["count", "median"])

par_pieces.round(0)

In [ ]:
verifier("la mediane des une piece", round(par_pieces.loc[1, "median"]) == 9533, "groupby('pieces'), colonne prix_m2")
verifier("la mediane des trois pieces", round(par_pieces.loc[3, "median"]) == 9667, "la ligne d'etiquette 3")

**Non.** Un une-pièce se vend 9 533 € le m², un trois-pièces 9 667, un
cinq-pièces 11 092. Le prix au m² **monte** avec la taille du logement,
exactement l'inverse de l'intuition.

Et regardez la colonne `count` avant d'aller plus loin : au-delà de six
pièces, les effectifs tombent à quelques dizaines, puis à quelques unités. La
médiane des onze-pièces est calculée sur **une seule vente**. C'est le réflexe
du cours 3.1 : l'effectif d'abord, le commentaire ensuite.

### Exercice 9 — L'explication

Si les grands appartements sont plus chers au m², ce n'est peut-être pas parce
qu'ils sont grands. C'est peut-être parce qu'ils ne sont pas là où sont les
petits.

Calculez `part_grands` : pour chaque arrondissement, le pourcentage de ventes
qui portent sur un logement de **4 pièces ou plus**. Affichez le résultat trié.

> **Rappel.** Deux `groupby` : un sur les seules ventes de 4 pièces et plus,
> un sur toutes. `size()` compte les lignes de chaque paquet. Diviser une
> Series par une autre met en face les étiquettes identiques — ici les numéros
> d'arrondissement — et vous n'avez rien à aligner vous-même.

In [ ]:
grands = ventes.query("pieces >= 4").groupby("arrondissement").size()
toutes = ventes.groupby("arrondissement").size()

part_grands = (100 * grands / toutes).round(1)

print(part_grands.sort_values(ascending=False))

In [ ]:
verifier("le 16e", part_grands.loc[16] == 33.5, "les ventes de 4 pieces et plus du 16e, divisees par toutes ses ventes")
verifier("le 18e", part_grands.loc[18] == 7.8, "meme calcul : une division de deux Series, etiquette par etiquette")

**33,5 % dans le 16e, 7,8 % dans le 18e.** Les grands appartements sont
concentrés dans les arrondissements chers. Leur prix au m² élevé vient donc
pour partie de **là où ils sont**, pas de leur taille.

C'est le premier exemple d'un phénomène que vous retrouverez partout : deux
variables avancent ensemble parce qu'une troisième les commande toutes les
deux. Ici, l'arrondissement commande à la fois la taille des logements et leur
prix. La suite du bloc 3 s'occupera de démêler ce genre de nœud ; pour
aujourd'hui, il suffit de savoir le repérer.

La cellule suivante montre la composition complète, à exécuter telle quelle.
`normalize="index"` transforme les effectifs de chaque ligne en parts de cette
ligne, de sorte que chaque arrondissement totalise 100 %.

In [ ]:
taille = ventes["pieces"].clip(upper=4)   ## tout ce qui depasse 4 devient 4 : la colonne "4" se lit "4 et plus"
composition = 100 * pd.crosstab(ventes["arrondissement"], taille, normalize="index")

composition.round(1)

---

## Partie 4 — L'estimateur

Tout est dans `fiche`. Il ne reste qu'à aller chercher la ligne d'un
arrondissement et à multiplier ses trois repères par une surface.

`fiche.loc[11]` rend **la ligne entière** du 11e, sous forme de Series : ses
étiquettes sont les noms de colonnes, donc `ligne["median"]` va chercher la
médiane. Exécutez la cellule, puis changez les deux premières lignes.

In [ ]:
arrondissement = 11    ## changez le numero
surface = 45           ## changez la surface, en m2

ligne = fiche.loc[arrondissement]
bas = ligne["q25"] * surface
milieu = ligne["median"] * surface
haut = ligne["q75"] * surface

print(f"{surface} m2 dans le {arrondissement}e arrondissement")
print(f"  le plus probable : {milieu:,.0f} euros".replace(",", " "))
print(f"  fourchette       : {bas:,.0f} a {haut:,.0f} euros".replace(",", " "))
print(f"  calcule sur {ligne['count']:.0f} ventes de 2024")

**442 238 €, entre 387 804 et 495 000.** C'est l'outil que la
direction demandait, et il tient en six lignes parce que tout le travail était
dans la fiche.

### Exercice 10 — L'épreuve du réel

Un estimateur qu'on ne confronte à rien ne vaut rien.

Choisissez un appartement que vous pouvez vérifier : le vôtre, celui d'un
proche, ou n'importe lequel. Faites tourner l'estimateur dessus. Puis allez
chercher **une vraie annonce** dans cet arrondissement, à surface comparable,
sur n'importe quel site d'agence.

En commentaire, dans la même cellule : le prix demandé, et si l'annonce tombe
dans la fourchette. Si elle en sort, écrivez ce qui pourrait l'expliquer.

> **Rappel.** Reprenez les lignes ci-dessus en changeant `arrondissement` et
> `surface`. `fiche.loc[numero]` pour la ligne, puis les trois colonnes.

In [ ]:
arrondissement = 18
surface = 30

ligne = fiche.loc[arrondissement]
print(round(ligne["q25"] * surface), "a", round(ligne["q75"] * surface),
      "| le plus probable :", round(ligne["median"] * surface))

# Annonce relevee sur un site d'agence : 30 m2 rue Ordener (18e), 295 000 euros.
# Elle tombe dans la fourchette, dans son quart superieur. Explications possibles :
# un etage eleve avec ascenseur, une renovation recente, ou simplement un prix
# affiche que personne ne paiera. L'estimateur ne tranche pas entre ces raisons :
# il dit seulement que ce prix n'a rien d'anormal pour ce quartier.

### Les vingt fourchettes, sur une image

Un point pour la médiane, une barre pour la moitié centrale. `plt.errorbar`
dessine exactement ça : `xerr` prend deux listes, de combien s'étendre à
gauche et de combien à droite du point. Exécutez la cellule.

In [ ]:
f = fiche.sort_values("median")
etiquettes = [f"{a}e" for a in f.index]

plt.figure(figsize=(7, 6))
plt.errorbar(f["median"], etiquettes,
             xerr=[f["median"] - f["q25"], f["q75"] - f["median"]],   ## a gauche, a droite
             fmt="o", color="#2878B5", ecolor="#F2B5B3", elinewidth=7, capsize=0)
plt.xlabel("prix au m2 (euros)")
plt.title("Mediane et moitie centrale des prix, par arrondissement (2024)")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

Les points donnent le classement que vous connaissez déjà. **Les barres sont
l'information nouvelle**, et elles racontent autre chose : celle du 6e est
presque trois fois plus longue que celle du 20e. Annoncer « le 6e est à
14 583 € le m² » est à peu près aussi utile que d'annoncer la température
moyenne d'un pays.

Regardez aussi les recouvrements. Le 16e est **2 000 € au-dessus du
12e** sur la médiane, soit près d'un quart de plus. Et pourtant leurs barres se
chevauchent, et **12 % des ventes du 12e se font plus cher que la
médiane du 16e**. Un classement de médianes ne dit rien de deux biens pris un
par un.

Cette question — un écart entre deux groupes est-il solide, ou tient-il à la
dispersion et au hasard ? — est exactement celle du cours suivant, et de
l'exercice qui l'accompagne.

---

## Pour conclure

Complétez cette cellule de texte (double-clic pour l'éditer) en trois phrases,
avec vos chiffres :

- Pour un 40 m² dans l'arrondissement que j'ai choisi, l'estimateur annonce entre … et … €.
- L'arrondissement où les prix sont les plus dispersés est le …, avec un coefficient de variation de … ; le plus homogène est le … .
- Ce que la médiane seule ne me disait pas : …

## Ce que vous avez fait

Vous êtes parti d'un classement de vingt médianes et vous en avez fait un
outil qui annonce une fourchette. En chemin :

- vous avez remplacé votre boucle de six lignes par un `groupby` d'une ligne, en sachant ce qu'il y a dedans ;
- vous avez mesuré la dispersion, et vu qu'elle n'est pas la même partout ;
- vous avez trouvé que 12 % des ventes portent 38 % de l'argent ;
- vous avez démonté une intuition de métier — le studio n'est pas plus cher au m² — et trouvé pourquoi.

| Vous avez utilisé | Pour |
|---|---|
| `describe()`, `mean()`, `median()` | la photo d'ensemble, et l'écart entre les deux centres |
| `groupby(...)[...].median()` | vingt médianes en une ligne |
| `groupby(...).size()`, `idxmin()` | compter par paquet, trouver le plus petit |
| `agg(["count", "mean", "median", "std"])` | quatre indicateurs d'un coup |
| `quantile(0.25)`, `quantile(0.75)` | les bornes de la moitié centrale |
| `fiche["q25"] = groupby(...)` | une colonne calculée, alignée par étiquette |
| `std() / mean()` | le coefficient de variation, qui se compare d'un groupe à l'autre |
| `query()`, `sum()`, `len()` | la concentration : 12 % des ventes, 38 % des euros |
| une Series divisée par une autre | des parts par arrondissement, sans aligner à la main |
| `crosstab(..., normalize="index")` | la composition complète, ligne par ligne |
| `fiche.loc[numero]`, f-string | l'estimateur |
| `plt.errorbar` | vingt fourchettes sur une image |

> ⚠️ **Avant de fermer l'onglet :** vérifiez que votre notebook est bien
> enregistré dans votre Drive.